# Chapter 1 — What Is an Embedding?

**Book alignment:** Embeddings From First Principles, Chapter 1

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** Does a real sentence encoder place a claim and its
*negation* almost on top of each other — i.e. does the geometry encode *aboutness* far more
than *assertion* — and is that visible in the committed RELATE measurement (Wave 1)?

Two cells build the identifier→embedding distinction in NumPy. The rest load the frozen
measured artifacts under `experiments/embeddings-from-first-principles/wave1/artifacts/`
and check the chapter's claims against them — no model download, no network.

In [ ]:
from pathlib import Path
import json
import numpy as np


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
ART = ROOT / "experiments" / "embeddings-from-first-principles" / "wave1" / "artifacts"
rc = json.loads((ART / "relation-cosine-by-type.json").read_text())
print("Wave 1 relation-cosine artifact — models measured:", ", ".join(rc["models"]))

## 1. Identifier vs embedding — the ladder in five lines of NumPy

An **identifier** supports equality and nothing else: one-hot vectors make every pair
equidistant. An **embedding** places objects so that geometry stands in for a relationship
— here, two crude features already separate a pet from a vehicle.

In [ ]:
# identifiers: one-hot. every pair is sqrt(2) apart -- no graded structure.
onehot = np.eye(3)
d_id = np.linalg.norm(onehot[:, None] - onehot[None, :], axis=-1)
assert np.allclose(d_id[np.triu_indices(3, 1)], np.sqrt(2))

# a 2-feature representation: (can_fly, is_pet).
feat = np.array([[0.0, 1.0],   # cat
                 [0.0, 1.0],   # dog
                 [1.0, 0.0]])  # airplane
d_feat = np.linalg.norm(feat[:, None] - feat[None, :], axis=-1)

print("identifier pair distances:", d_id[np.triu_indices(3, 1)].round(3), " (all equal)")
print("feature    pair distances:", d_feat[np.triu_indices(3, 1)].round(3),
      " (cat~dog close; both far from airplane)")
assert d_feat[0, 1] < d_feat[0, 2]
print("\nidentifier -> embedding is the move from 'these differ' to 'these differ in graded ways'")

## 2. The objective decides what is preserved — measured on RELATE

RELATE pairs are labelled by relationship. Embed both items of each typed pair and take the
mean cosine. Read the relations that *should* be low.

In [ ]:
M = "bge-large"
means = {rel: v["mean"] for rel, v in rc["models"][M]["by_relation"].items()}
for rel, m in sorted(means.items(), key=lambda kv: -kv[1]):
    print(f"  {rel:20} {m:.3f}")

order = [rel for rel, _ in sorted(means.items(), key=lambda kv: -kv[1])]
# 'Acme acquired Beta' vs 'Beta acquired Acme' scores at the very top with true paraphrases
assert set(order[:2]) == {"equivalent", "relation-swap"}
assert means["relation-swap"] > 0.96
assert abs(means["negation"] - means["contradiction"]) < 0.02
assert means["unrelated"] == min(means.values())
print(f"\nrelation-swap ({means['relation-swap']:.2f}) ranks with 'equivalent' at the top;")
print(f"negation ({means['negation']:.2f}) sits on top of contradiction ({means['contradiction']:.2f}).")
print("aboutness is encoded strongly; assertion (polarity, argument order) barely at all")

## 3. The paraphrase-vs-negation gap is small — and its sign is model-dependent

If a sentence's *negation* is as close as its *paraphrase*, the model encodes what the
sentence is *about*, not what it *asserts*. The gap is tiny and does not even agree in sign
across encoders.

In [ ]:
gaps = {}
for m, blob in rc["models"].items():
    br = blob["by_relation"]
    gaps[m] = br["paraphrase"]["mean"] - br["negation"]["mean"]
    print(f"  {m:14} paraphrase - negation = {gaps[m]:+.3f}")

spread = max(gaps.values()) - min(gaps.values())
print(f"\nspread across models: {spread:.3f}")
assert spread > 0.05                        # the ranking is not stable across encoders
assert min(gaps.values()) < 0.065           # even the largest gap is within noise
print("the gap is tiny for every encoder and its sign is not stable across models")

## 4. A vector without its space record is a list of numbers you chose to forget

Every cosine above is a fact about *one* learned transformation under *one* metric — not a
fact about meaning.

In [ ]:
space_record = {
    "model":         "BAAI/bge-large-en-v1.5",
    "dimension":     1024,
    "normalization": "l2",
    "objective":     "contrastive retrieval (query <-> passage)",
    "similar_means": "same topic / same entities; polarity, argument order and time are near the noise floor",
}
print(json.dumps(space_record, indent=2))
assert space_record["similar_means"].startswith("same topic")
print("\ngeometry is evidence about a representation, not permission to use it")

## What we earned

An embedding is the output of a specific learned transformation under an objective — not a
container of meaning. On the frozen RELATE measurement, reversing *who acquired whom*
barely moves the vector, and a sentence sits about as close to its negation as to its
paraphrase: the objective encoded *aboutness*, not *assertion*.

**Notebook 02 / Chapter 2** builds a tiny space by hand and watches semantic questions turn
into distance, direction, and angle — and marks where that translation starts to leak.